---
title: Estimación de la Severidad tras un incendio con imágenes Sentinel
subject: Ejercicio
subtitle: Ejercicio que muestra cómo obtener productos biofísicos así como calcular el NBR antes y después de un incendio forestla
authors:
  - name: Héctor Nieto
    affiliations:
      - Instituto de Ciencias Agrarias, ICA
      - CSIC
    orcid: 0000-0003-4250-6424
    email: hector.nieto@ica.csic.es
  - name: Radoslaw Guzinski
    affiliations:
      - DHI
    orcid: 0000-0003-0044-6806
  - name: Benjamin Mary
    affiliation:
      - Instituto de Ciencias Agrarias
      - CSIC
    orcid: 0000-0003-0815-842X
label: nb-senet
license: CC-BY-SA-4.0
keywords: Sentinel-2, NBR, biophysical traits
myst:
  enable_extensions: ["deflist", "attrs_block", "attrs_inline"]
jupytext:
  text_representation:
    extension: .md
    format_name: myst
    format_version: 0.13
    jupytext_version: 1.19.1
kernelspec:
  display_name: Python 3 (ipykernel)
  language: python
  name: python3
---

# Introducción
En este ejercicio vamos a descargar unas pocas imágenes Sentinel-2 antes y después de un incendio forestal para después generar los productos biofísicos mediante un algoritmo propio, así como el NBR.

Usaremos para ello usando la librería Python de [openEO](https://openeo.org/) para generar los promedios zonales de cada imagen de reflectividad y a parti de ahí continuar con el procesamiento en local.

Este cuaderno puede ejecutarse en el [Jupyterhub de Copernicus Dataspace](https://jupyterhub.dataspace.copernicus.eu), en cuyo caso no se realizan descargas locales de datos, ya que tanto los datos como el entorno de ejecución están en CDSE y se mantienen en tu cuenta.

:::{warning} Atención
Si estás usando el entorno de CDSE debes seleccionar uno de los kernels con GDAL instalado, p. ej. "Geo science".
:::

::::{seealso} Ver también
:::{table} Características de la misión Sentinel-2
:label: s2
Plataformas | Rango espectral     | Número de bandas | Resolución espacial | Resolución temporal
:---        | :---                | :---             | :---                | :---            
A, B, C     | Visible, NIR, SWIR  | 10 (13)          | 10 -- 20 m          | 5 -- 10 días
:::
::::

# Activación de librerías
Primero comprobamos que el Sen-ET Toolbox esté instalado (y lo instalamos si es necesario) y luego importamos todos los paquetes necesarios.   

In [ ]:
try:
    import senet_toolbox
    print("senet_toolbox importado correctamente")
except ModuleNotFoundError:
    print("Falta la librería senet_toolbox, instalando desde Git")
    !pip install senet_toolbox@git+https://github.com/DHI/Sen-ET-OpenEO-toolbox.git

In [ ]:
from IPython.display import display
import datetime as dt
from pathlib import Path
import datetime as dt
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import plotly.graph_objects as go
import openeo
import rioxarray
import xarray as xr
from ipywidgets import interact, interactive, fixed, widgets
from ipyleaflet import Map, basemaps, basemap_to_tiles, DrawControl, Polygon
import multiprocessing as mp
import Py6S as sixs
from pypro4sail import machine_learning_regression as inv
from pyTSEB import meteo_utils as met
from sklearn.ensemble import RandomForestRegressor as rf_sklearn
from statsmodels.tsa.seasonal import MSTL
from senet_toolbox.utils.raster_utils import save_raster, default_profile
from senet_toolbox.utils import visualization
from senet_toolbox.workflows.collect_input_data import wait_and_download
import warnings
print("Librerías importadas correctamente, puedes continuar")

# Descargar imágenes Sentinel-2
Descargaremos las imágenes Sentinel-2 más cercanas a la ocurrencia de un incendio o perturbación

## Seleccionar el Área de Interés
Para mantener los datos organizados y facilitar el procesamiento de series temporales, los datos de entrada y salida se guardan en carpetas de Área de Interés (AOI). Todos los datos dentro de una carpeta AOI tienen la misma extensión y cuadrícula.

En la celda siguiente, selecciona la ubicación donde deseas almacenar los datos y el nombre del AOI. Al ejecutar en el Jupyterhub de CDSE, se recomienda mantenerlo dentro de `./mystorage/301-biophysical`, de lo contrario los datos se borrarán entre sesiones.

Si estás configurando un nuevo AOI, tendrás que subir una capa de polígonos con las zonas de interés. Se recomienda seleccionar AOIs de pequeñas (unos pocos kilómetros) para agilizar el procesado y no usar los cŕeditos gratuitos rápidamente.

In [ ]:
data_dir = "./mystorage/303-severity"
aoi_name = "agramon"
aoi_data_dir = Path(data_dir) / aoi_name
if not aoi_data_dir.is_dir():
    aoi_data_dir.mkdir()

In [ ]:
# Dibuja o visualiza la extensión del AOI al configurar uno nuevo
map, bboxs = visualization.select_aoi(aoi_data_dir)
map

## Seleccionar el rango de fechas
En la siguiente celda selecciona una fecha de inicia (poco antes del incendio) y una fecha final (poco después del incendio)

In [ ]:
w_date_ini = widgets.DatePicker(
    description='Fecha inicial:',
    value=dt.datetime(2020, 7, 19),
    disabled=False
)
w_date_end = widgets.DatePicker(
    description='Fecha final:',
    value=dt.datetime(2020, 7, 24),
    disabled=False
)
display(w_date_ini, w_date_end)

## Conectarse al backend de OpenEO

Las imágenes de Sentinel-3 y Sentinel-2, el mapa de cobertura del suelo Worldcover y el Modelo Digital de Elevación de Copernicus se descargan desde la interfaz OpenEO de CDSE. Ejecuta la celda siguiente para autenticarte en OpenEO.

**Nota:** Es posible que debas hacer clic en un enlace de autenticación que aparecerá y seguir las instrucciones.

In [ ]:
connection = openeo.connect("https://openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()

## Descargar series temporales de Sentinel-2
Descargaremos la serie temporal de los datos de reflectividad de Sentinel-2, promediados espacialmente según los polígonos que hemos introducido anteriormente.

:::{important} Importante
Este proceso puede llevar un tiempo, ten paciencia o hazlo durante varios días.
:::

In [ ]:
input_dir = aoi_data_dir / "input"
s2l2a_filename = f"s2_{w_date_ini.value:%Y%m%d}_{w_date_end.value:%Y%m%d}_data.nc"
aoi = dict(zip(["west", "south", "east", "north"], bboxs[-1]))

COLLECTION = "SENTINEL2_L2A"
S2_BANDS = ["B02", "B03", "B04", "B05", "B06",
            "B07", "B08", "B8A", "B11", "B12"]

BANDS = ["SCL", "AOT", "WVP"] + S2_BANDS

s2_path = input_dir / s2l2a_filename
   
if not s2_path.exists(): 
    if not input_dir.is_dir():
        input_dir.mkdir(parents=True)
        
    connection.authenticate_oidc_refresh_token()
    s2l2a = connection.load_collection(
        COLLECTION,
        spatial_extent=aoi,
        temporal_extent=[w_date_ini.value,
                         w_date_end.value + dt.timedelta(1)],
        bands=BANDS
    )
    
    # Select the "SCL" band from the data cube
    s2l2a = s2l2a.mask(s2l2a.band("SCL") < 4).mask(s2l2a.band("SCL") > 5)
    # Resample to 20 m and warp to geographic projection
    s2l2a = s2l2a.resample_spatial(
        resolution=20, method="average").resample_spatial(projection=4326)
    
    job = s2l2a.create_job(out_format="netcdf")
    job.start()
    wait_and_download(job, s2_path, poll_interval=60)
    
print("Terminado con todas las tareas, puedes proseguir a la siguiente celda")

:::{attention} Atención
Si has recibido algún error tipo `ConcurrentJobLimit: Job was not started because concurrent job limit (30) is reached` ejecuta la celda [](#remove-jobs) para eliminar todos los trabajos pendientes de la nube y reintentar
:::

# Calcula el NBR para cada fecha y estima los productos biofísicos
Al contrario que en la práctica [301a](301a-ES_parametros_biofisicos.ipynb), vamos a generar nosotros mismos los productos biofísicos mediante la construcción de base de datos de parámetros biofísicos y espectros correspondientes simulados con ProspectD+4SAIL.

:::{seealso} Ver también
Puedes volver a recordar detalles sobre estos modelos en el cuaderno digital [102](./102-ES_espectro_vegetacion.ipynb)

Tambiés puedes leer el algoritmo original en [Weiss y Baret (2016). S2ToolBox Level 2 products: LAI, FAPAR, FCOVER Version 1.1](http://step.esa.int/docs/extra/ATBD_S2ToolBox_L2B_V1.1.pdf)
:::



In [ ]:
out_dir = aoi_data_dir / "output"

S2_BANDS = ["B02", "B03", "B04", "B05", "B06",
            "B07", "B08", "B8A", "B11", "B12"]

NBR_BANDS = ["B8A", "B12"]

# Puedes subir este valor (p.e. 10000 o 50000) para tener más robustez en las estimaciones a coste de un mayor tiempo de procesado
N_SIMULATIONS = 5000
# Uso de computación en paralelo
N_JOBS = 2

# Selecciona entre "Cab", "Car", "Cm", "Cw", "Ant", "Cbrown", "LAI", "leaf_angle"
OBJ_PARAM_NAMES = ["Cab", "Cw", "Ant", "LAI"]


# "Cab", "Car", "Cm", "Cw", "Ant", "Cbrown", "LAI", "leaf_angle"
# Path to the pyPro4SAIL soil library and SRF library
SOIL_LIBRARY = Path(inv.__file__).parent / "spectra" / "soil_spectral_library"
SRF_LIBRARY = Path(inv.__file__).parent / "spectra" / "sensor_response_functions"
WLS_SIM = np.arange(400, 2501)
SATELLITE = "2A"
ACQ_TIME = 10.5


def get_diffuse_radiation_6S(aot, wvp, sza, saa, date,
                             altitude=0.1, wls_step=10, n_jobs=1):
    warnings.simplefilter("ignore")
        
    s = sixs.SixS()
    s.atmos_profile = sixs.AtmosProfile.PredefinedType(
        sixs.AtmosProfile.MidlatitudeSummer)

    s.aeroprofile = sixs.AeroProfile.PredefinedType(
        sixs.AeroProfile.Continental)

    s.ground_reflectance = sixs.GroundReflectance.HomogeneousLambertian(0)

    if np.isfinite(wvp) and wvp > 0:
        s.atmos_profile = sixs.AtmosProfile.UserWaterAndOzone(wvp, 0.9)

    if np.isfinite(aot) and aot > 0:
        s.aot550 = aot

    s.geometry.solar_z = sza
    s.geometry.solar_a = saa
    s.geometry.view_z = 0
    s.geometry.view_a = 0
    s.geometry.day = date.day
    s.geometry.month = date.month

    s.altitudes.set_target_custom_altitude(altitude)
    s.wavelength = sixs.Wavelength(0.4, 2.5)

    wls = np.arange(400, 2501)
    wls_sim = np.arange(400, 2501, wls_step)

    wv, res = sixs.SixSHelpers.Wavelengths.run_wavelengths(s,
                                                           wls_sim / 1000.,
                                                           verbose=False,
                                                           n=n_jobs)

    eg_d = np.array(sixs.SixSHelpers.Wavelengths.extract_output(res,
                                                                'diffuse_solar_irradiance'))

    eg_s = np.array(sixs.SixSHelpers.Wavelengths.extract_output(res,
                                                                'direct_solar_irradiance'))

    eg_d = np.maximum(eg_d, 0)
    eg_s = np.maximum(eg_s, 0)
    skyl = np.full_like(wls, np.nan, dtype=np.float64)
    # Fill the diffuse values into a full wavelenght array
    valid = np.in1d(wls, wls_sim, assume_unique=True)
    skyl[valid] = eg_d / (eg_d + eg_s)
    # Fill nans by linear interpolation
    nans, x = np.isnan(skyl), lambda z: z.nonzero()[0]
    skyl[nans] = np.interp(x(nans), x(~nans), skyl[~nans])

    return skyl


def build_soil_database(soil_albedo_factor,
                        soil_library=SOIL_LIBRARY):
    soil_library = Path(soil_library)
    n_simulations = np.size(soil_albedo_factor)
    soil_files = list(soil_library.glob('jhu.*spectrum.txt'))
    n_soils = len(soil_files)
    soil_spectrum = []
    for soil_file in soil_files:
        r = np.genfromtxt(soil_file)
        soil_spectrum.append(r[:, 1])

    multiplier = int(np.ceil(float(n_simulations / n_soils)))
    soil_spectrum = np.asarray(soil_spectrum * multiplier)
    soil_spectrum = soil_spectrum[:n_simulations]
    soil_spectrum = soil_spectrum * soil_albedo_factor.reshape(-1, 1)
    soil_spectrum = np.clip(soil_spectrum, 0, 1)
    soil_spectrum = soil_spectrum.T
    return soil_spectrum


def biophysical_retrieval(date, lat, lon, refl, aot, tcwv, 
                          n_simulations=40000, n_jobs=-1):
    warnings.simplefilter("ignore")        
    if n_jobs <= 0:
        n_jobs = mp.cpu_count()
    
    start_time = dt.datetime.now()    
    doy = date.dayofyear
    warnings.simplefilter("ignore")
    params_orig = inv.build_prosail_database(n_simulations,
                                             distribution=inv.SALTELLI_DIST)

    scikit_regressor_opts = {"n_estimators": 100,
                             "min_samples_leaf": 1,
                             "n_jobs": n_jobs}

    # Get Solar angles
    sza, saa = met.calc_sun_angles(lat,
                                   lon,
                                   lon,
                                   doy,
                                   ACQ_TIME)
    vza = 0
    # Running 6S for estimation of diffuse/direct irradiance
    try:
        skyl = get_diffuse_radiation_6S(
                aot, tcwv, sza, saa, date,
                altitude=0.1, wls_step=100)
    except:
        print("6S missing, assuming a constant diffuse ratio")
        skyl = 0.2
        
    # Stack spectral bands
    srf = []
    srf_file = SRF_LIBRARY / f'Sentinel{SATELLITE}.txt'
    srfs = np.genfromtxt(srf_file, dtype=None, names=True)
    for band in S2_BANDS:
        srf.append(srfs[band])

    # Builing standard soil database
    soil_spectrum = build_soil_database(params_orig["bs"])
    # Building {np.size(params_orig['bs'])} PROSPECTD+4SAIL simulations

    rho_canopy_vec, params = inv.simulate_prosail_lut(
        params_orig,
        WLS_SIM,
        soil_spectrum,
        skyl=skyl,
        sza=np.full_like(params_orig["LAI"], sza),
        vza=np.full_like(params_orig["LAI"], vza),
        psi=np.zeros_like(params_orig["LAI"]),
        srf=srf,
        outfile=None,
        calc_FAPAR=False,
        reduce_4sail=True)

    params = pd.DataFrame(params)
    reg = rf_sklearn(**scikit_regressor_opts)
    # Apply model to Landsat image
    out_dict = {}
    # Should be — flatten spatial dims, predict, then reshape
    dims = refl.shape[1:]  
    refl = refl.reshape(refl.shape[0], -1).T
    valid_mask = np.isfinite(refl).all(axis=1)
    refl = refl[valid_mask]
    for i, param in enumerate(OBJ_PARAM_NAMES):
        reg = reg.fit(rho_canopy_vec, params[param])
        output = np.full(valid_mask.shape, np.nan)
        output[valid_mask] = reg.predict(refl)
        min_value = inv.prosail_bounds[param][0]
        max_value = inv.prosail_bounds[param][1]
        output = np.clip(output, min_value, max_value)
        out_dict[param] = output.reshape(dims)

    elapsed = (dt.datetime.now() - start_time).total_seconds()
    print(f"Finalizado {date} en {elapsed :.0f} segundos")
    return out_dict


def calc_vi(xarr, b1, b2):
    vi = (xarr[b1] - xarr[b2]) / (xarr[b1] + xarr[b2])
    return vi
    

def xarr_2_tif(result_da, output_file):
    result_da = result_da.rio.write_crs("EPSG:4326")
    result_da = result_da.rio.set_spatial_dims(x_dim="x", y_dim="y")
    result_da.rio.to_raster(output_file)    
    result_da.attrs.pop("grid_mapping", None)
    result_da.rio.to_raster(output_file, **default_profile())
    
    
def nc_to_biodata(ds, date_obj, out_dir): 
    ds = ds.sel(t=date_obj)   
    date_obj = pd.to_datetime(str(date_obj))
    date_str = date_obj.strftime('%Y%m%d')
    nodata = (ds[S2_BANDS].to_array() < 0).any(dim="variable")
    # Scale to get the reflectance values
    refl = ds[S2_BANDS].where(~nodata) / 10000
    nbr = calc_vi(refl, *NBR_BANDS)
    output_file = out_dir / f"s2_{date_str}_NBR.tif"
    xarr_2_tif(nbr, output_file)
    # Get mean AOT and WVP after scaling the values
    aot = np.nanmean(ds["AOT"].where(~nodata) / 1000)
    wvp = np.nanmean(ds["WVP"].where(~nodata) / 1000)
    # Convert xarray Dataset → (bands, y, x) numpy array
    refl_np = np.stack([refl[b].values for b in S2_BANDS], axis=0)
    output_dict = biophysical_retrieval(
        date_obj, lat, lon, refl_np, aot, wvp, n_simulations=N_SIMULATIONS, n_jobs=N_JOBS)
    
    for param, output in output_dict.items():
        output_file = out_dir / f"s2_{date_str}_{param}.tif"
        output = xr.DataArray(output,
                              dims=["y", "x"],
                              coords={"y": ds.y, "x": ds.x},
                              name=param
                             )
        xarr_2_tif(output, output_file)
        
        
if not out_dir.is_dir():
    out_dir.mkdir(parents=True)
    
ds = xr.open_dataset(s2_path)
# Assign CRS — NetCDF rarely embeds it
ds = ds.rio.write_crs("EPSG:4326")  # WGS84 for lat/lon data; adjust if projected
# Get mean scene coordinates
lon = ds.x.mean().item()
lat = ds.y.mean().item()
for date_obj in ds.t.values:
    print(f"Extrayendo NBR y rasgos biofísicos para {date_obj}")
    nc_to_biodata(ds, date_obj, out_dir)

print("Calculado el NBR y los rasgos biofísicos para todas las fechas")

# Analiza la severidad
Vamos a calcular ahora el dNBR 

:::{math}
dNBR = NBR_{pre} - NBR_{post}
:::

y de manera similar para el LAI, la pérdida de LAI

:::{math}
dLAI = LAI_{pre} - LAI_{post}
:::

y el porcentaje de pérdida

:::{math}
dLAI_{rel} = \frac{dLAI}{LAI_{pre}}
:::
Entre la fecha antes del incendio y la fecha después.

Para ello usa algún visor para escoger una imagen libre de nubes antes del incendio y otra imagen también libre de nubes después del incendio

In [ ]:
w_date_pre = widgets.DatePicker(
    description='Fecha previa:',
    value=dt.datetime(2020, 7, 19),
    disabled=False
)
w_date_post = widgets.DatePicker(
    description='Fecha posterior:',
    value=dt.datetime(2020, 7, 24),
    disabled=False
)
display(w_date_pre, w_date_post)

In [ ]:
date_pre = f"{w_date_pre.value:%Y%m%d}"
date_post = f"{w_date_post.value:%Y%m%d}"

print(f"Calculando dNBR entre {date_pre} y {date_post}")
file_path = out_dir / f"s2_{date_pre}_NBR.tif"
src = rasterio.open(file_path)
profile = src.profile
pre = src.read(1)
src.close()

file_path = out_dir / f"s2_{date_post}_NBR.tif"
src = rasterio.open(file_path)
post = src.read(1)
src.close()

valid = np.logical_and(np.isfinite(pre), np.isfinite(post))
output = np.full_like(pre, np.nan)
output[valid] = pre[valid] - post[valid]
out_file = out_dir / f"s2_{date_post}_dNBR.tif"
with rasterio.open(out_file, 'w', **profile) as dst:
    dst.write(output, 1)

out_file = out_dir / f"s2_{date_post}_dNBR_rel.tif"
with rasterio.open(out_file, 'w', **profile) as dst:
    dst.write(output / pre, 1)

print(f"Calculando porcentaje de pérdida de LAI entre {date_pre} y {date_post}")
file_path = out_dir / f"s2_{date_pre}_LAI.tif"
src = rasterio.open(file_path)
profile = src.profile
pre = src.read(1)
src.close()

file_path = out_dir / f"s2_{date_post}_LAI.tif"
src = rasterio.open(file_path)
post = src.read(1)
src.close()

valid = np.logical_and(np.isfinite(pre), np.isfinite(post))
output = np.full_like(pre, np.nan)
output[valid] = 100 * (pre[valid] - post[valid]) / pre[valid]
output[valid] = (pre[valid] - post[valid]) 
out_file = out_dir / f"s2_{date_post}_dLAI.tif"
with rasterio.open(out_file, 'w', **profile) as dst:
    dst.write(output, 1)

out_file = out_dir / f"s2_{date_post}_dLAI_rel.tif"
with rasterio.open(out_file, 'w', **profile) as dst:
    dst.write(output / pre, 1)
    
print(f"Calculados indicadores de severidad entre {date_pre} y {date_post}")

## Visualiza los productos

### dNBR

In [ ]:
file_path = out_dir / f"s2_{date_post}_dNBR.tif"
visualization.show_raster_map(
    file_path,
    cmap="RdYlGn_r"
)


### Diferencia de LAI

In [ ]:
file_path = out_dir / f"s2_{date_post}_dLAI.tif"
visualization.show_raster_map(
    file_path,
    cmap="RdYlGn_r"
)

### Porcentaje de pérdida de LAI

In [ ]:
file_path = out_dir / f"s2_{date_post}_dLAI_rel.tif"
visualization.show_raster_map(
    file_path,
    cmap="RdYlGn_r"
)

# Ejercicio
1. Genera tu propia capa el incendio que quieras evaluar
2. Calcula los productos biofísicos y el NBR
3. Evalúa la severidad
4. Genera un pequeño informe de 1-2 págines detallando conclusiones sobre las tendencias observadas
5. Exporta todo el cuaderno en formato pdf

(remove-jobs)=
# Eliminar trabajos actuales
Si has recibido algún error tipo `ConcurrentJobLimit: Job was not started because concurrent job limit (30) is reached` ejecuta esta celda para eliminar todos los trabajos pendientes de la nube y reintentar

In [ ]:
jobs = connection.list_jobs()
for job in jobs:
    if job["status"] == "finished":
        continue
    
    print(f"Borrando trabajo {job}")
    job = connection.job(job["id"])    
    job.delete()

print("Todos los trabajos pendientes en cola eliminados, puedes volver a procesar los productos")